# Импорт данных с экономией памяти: новая поставка


## Принципы импорта

- импортируем только актуальные CSV из `src`;
- не читаем технический первый столбец и явно ненужные поля из документации;
- сразу приводим даты, категории и числовые типы, чтобы не держать лишние `object` в памяти;
- отдельно загружаем `stats__module_*`, потому что они уже нужны для target.


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

SRC_DIR = Path.cwd() / "src"
csv_kwargs = {"thousands": ",", "true_values": ["True"], "false_values": ["False"]}


In [2]:
# Базовые таблицы: пользователи, курсы, уроки, группы и тренинги.

users_df = pd.read_csv(
    SRC_DIR / "users.csv",
    usecols=[
        "id",
        "last_explainer_seen_→_course",
        "created_at",
        "updated_at",
        "type",
        "sign_in_count",
        "last_sign_in_at",
        "grade_id",
        "subscribed",
        "is_teacher",
        "timezone",
        "grade_changed_at",
        "xp",
        "d_wk_school_id",
        "d_wk_municipal_id",
        "d_wk_region_id",
        "wk_gender",
    ],
    **csv_kwargs,
)
users_df["created_at"] = pd.to_datetime(users_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["updated_at"] = pd.to_datetime(users_df["updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["last_sign_in_at"] = pd.to_datetime(users_df["last_sign_in_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["grade_changed_at"] = pd.to_datetime(users_df["grade_changed_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["id"] = pd.to_numeric(users_df["id"], errors="coerce").astype("Int32")
users_df["last_explainer_seen_→_course"] = pd.to_numeric(users_df["last_explainer_seen_→_course"], errors="coerce").astype("Int8")
users_df["sign_in_count"] = pd.to_numeric(users_df["sign_in_count"], errors="coerce").astype("Int16")
users_df["grade_id"] = pd.to_numeric(users_df["grade_id"], errors="coerce").astype("Int16")
users_df["xp"] = pd.to_numeric(users_df["xp"], errors="coerce").astype("Int32")
users_df["d_wk_school_id"] = pd.to_numeric(users_df["d_wk_school_id"], errors="coerce").astype("Int32")
users_df["d_wk_municipal_id"] = pd.to_numeric(users_df["d_wk_municipal_id"], errors="coerce").astype("Int32")
users_df["d_wk_region_id"] = pd.to_numeric(users_df["d_wk_region_id"], errors="coerce").astype("Int32")
users_df["wk_gender"] = pd.to_numeric(users_df["wk_gender"], errors="coerce").astype("Int8")
users_df["type"] = users_df["type"].astype("category")
users_df["timezone"] = users_df["timezone"].astype("category")
users_df["subscribed"] = users_df["subscribed"].astype("boolean")
users_df["is_teacher"] = users_df["is_teacher"].astype("boolean")

users_courses_df = pd.read_csv(
    SRC_DIR / "users_courses.csv",
    usecols=[
        "id",
        "user_id",
        "course_id",
        "state",
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_points",
        "wk_max_points",
        "wk_max_viewable_lessons",
        "wk_max_task_count",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],
    **csv_kwargs,
)
users_courses_df["created_at"] = pd.to_datetime(users_courses_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["updated_at"] = pd.to_datetime(users_courses_df["updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["access_finished_at"] = pd.to_datetime(users_courses_df["access_finished_at"], format="%d %b, %Y", errors="coerce")
users_courses_df["wk_officially_started_at"] = pd.to_datetime(users_courses_df["wk_officially_started_at"], format="%d %b, %Y", errors="coerce")
users_courses_df["wk_course_completed_at"] = pd.to_datetime(users_courses_df["wk_course_completed_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["id"] = pd.to_numeric(users_courses_df["id"], errors="coerce").astype("Int32")
users_courses_df["user_id"] = pd.to_numeric(users_courses_df["user_id"], errors="coerce").astype("Int32")
users_courses_df["course_id"] = pd.to_numeric(users_courses_df["course_id"], errors="coerce").astype("Int32")
users_courses_df["wk_points"] = pd.to_numeric(users_courses_df["wk_points"], errors="coerce").astype("Float32")
users_courses_df["wk_max_points"] = pd.to_numeric(users_courses_df["wk_max_points"], errors="coerce").astype("Float32")
users_courses_df["wk_max_viewable_lessons"] = pd.to_numeric(users_courses_df["wk_max_viewable_lessons"], errors="coerce").astype("Float32")
users_courses_df["wk_max_task_count"] = pd.to_numeric(users_courses_df["wk_max_task_count"], errors="coerce").astype("Float32")
users_courses_df["state"] = users_courses_df["state"].astype("category")

lessons_df = pd.read_csv(
    SRC_DIR / "lessons.csv",
    usecols=[
        "id",
        "course_id",
        "conspect_expected",
        "task_expected",
        "lesson_number",
        "wk_max_points",
        "wk_task_count",
        "wk_survival_training_expected",
        "wk_scratch_playground_enabled",
        "wk_attendance_tracking_enabled",
        "wk_video_duration",
        "wk_attendance_tracking_disabled_at",
    ],
    **csv_kwargs,
)
lessons_df["wk_attendance_tracking_disabled_at"] = pd.to_datetime(
    lessons_df["wk_attendance_tracking_disabled_at"], format="%d %b, %Y, %H:%M", errors="coerce"
)
lessons_df["id"] = pd.to_numeric(lessons_df["id"], errors="coerce").astype("Int32")
lessons_df["course_id"] = pd.to_numeric(lessons_df["course_id"], errors="coerce").astype("Int32")
lessons_df["lesson_number"] = pd.to_numeric(lessons_df["lesson_number"], errors="coerce").astype("Int16")
lessons_df["wk_max_points"] = pd.to_numeric(lessons_df["wk_max_points"], errors="coerce").astype("Float32")
lessons_df["wk_task_count"] = pd.to_numeric(lessons_df["wk_task_count"], errors="coerce").astype("Float32")
lessons_df["wk_video_duration"] = pd.to_numeric(lessons_df["wk_video_duration"], errors="coerce").astype("Float32")
lessons_df["conspect_expected"] = lessons_df["conspect_expected"].astype("boolean")
lessons_df["task_expected"] = lessons_df["task_expected"].astype("boolean")
lessons_df["wk_survival_training_expected"] = lessons_df["wk_survival_training_expected"].astype("boolean")
lessons_df["wk_scratch_playground_enabled"] = lessons_df["wk_scratch_playground_enabled"].astype("boolean")
lessons_df["wk_attendance_tracking_enabled"] = lessons_df["wk_attendance_tracking_enabled"].astype("boolean")

groups_df = pd.read_csv(
    SRC_DIR / "groups.csv",
    usecols=[
        "id",
        "lesson_id",
        "teacher_id",
        "starts_at",
        "duration",
        "state",
        "group_template_id",
        "video_available",
        "pupils_notified_at",
        "wk_actual_started_at",
        "wk_actual_finished_at",
        "wk_duration_actual",
    ],
    **csv_kwargs,
)
groups_df["starts_at"] = pd.to_datetime(groups_df["starts_at"], format="%d %b, %Y, %H:%M", errors="coerce")
groups_df["pupils_notified_at"] = pd.to_datetime(groups_df["pupils_notified_at"], format="%d %b, %Y, %H:%M", errors="coerce")
groups_df["wk_actual_started_at"] = pd.to_datetime(groups_df["wk_actual_started_at"], format="%d %b, %Y, %H:%M", errors="coerce")
groups_df["wk_actual_finished_at"] = pd.to_datetime(groups_df["wk_actual_finished_at"], format="%d %b, %Y, %H:%M", errors="coerce")
groups_df["id"] = pd.to_numeric(groups_df["id"], errors="coerce").astype("Int32")
groups_df["lesson_id"] = pd.to_numeric(groups_df["lesson_id"], errors="coerce").astype("Int32")
groups_df["teacher_id"] = pd.to_numeric(groups_df["teacher_id"], errors="coerce").astype("Int32")
groups_df["duration"] = pd.to_numeric(groups_df["duration"], errors="coerce").astype("Int16")
groups_df["group_template_id"] = pd.to_numeric(groups_df["group_template_id"], errors="coerce").astype("Int32")
groups_df["state"] = groups_df["state"].astype("category")
groups_df["video_available"] = groups_df["video_available"].astype("boolean")
groups_df["wk_duration_actual"] = groups_df["wk_duration_actual"].astype("boolean")

trainings_df = pd.read_csv(
    SRC_DIR / "trainings.csv",
    usecols=["id", "discipline_id", "time_limit", "published_at", "difficulty", "lesson_id", "task_templates_count"],
    **csv_kwargs,
)
trainings_df["published_at"] = pd.to_datetime(trainings_df["published_at"], format="%d %b, %Y, %H:%M", errors="coerce")
trainings_df["id"] = pd.to_numeric(trainings_df["id"], errors="coerce").astype("Int32")
trainings_df["discipline_id"] = pd.to_numeric(trainings_df["discipline_id"], errors="coerce").astype("Int32")
trainings_df["time_limit"] = pd.to_numeric(trainings_df["time_limit"], errors="coerce").astype("Int16")
trainings_df["difficulty"] = pd.to_numeric(trainings_df["difficulty"], errors="coerce").astype("Int8")
trainings_df["lesson_id"] = pd.to_numeric(trainings_df["lesson_id"], errors="coerce").astype("Int32")
trainings_df["task_templates_count"] = pd.to_numeric(trainings_df["task_templates_count"], errors="coerce").astype("Int8")


In [3]:
# Задачи, домашние задания и логи прохождения курса.

lesson_tasks_df = pd.read_csv(
    SRC_DIR / "lesson_tasks.csv",
    usecols=["id", "lesson_id", "task_id", "position", "task_required"],
    **csv_kwargs,
)
lesson_tasks_df["id"] = pd.to_numeric(lesson_tasks_df["id"], errors="coerce").astype("Int32")
lesson_tasks_df["lesson_id"] = pd.to_numeric(lesson_tasks_df["lesson_id"], errors="coerce").astype("Int32")
lesson_tasks_df["task_id"] = pd.to_numeric(lesson_tasks_df["task_id"], errors="coerce").astype("Int32")
lesson_tasks_df["position"] = pd.to_numeric(lesson_tasks_df["position"], errors="coerce").astype("Int16")
lesson_tasks_df["task_required"] = lesson_tasks_df["task_required"].astype("boolean")

homeworks_df = pd.read_csv(
    SRC_DIR / "homeworks.csv",
    usecols=["id", "resource_type", "resource_id", "homework_type"],
    **csv_kwargs,
)
homeworks_df["id"] = pd.to_numeric(homeworks_df["id"], errors="coerce").astype("Int32")
homeworks_df["resource_id"] = pd.to_numeric(homeworks_df["resource_id"], errors="coerce").astype("Int32")
homeworks_df["homework_type"] = pd.to_numeric(homeworks_df["homework_type"], errors="coerce").astype("Int8")
homeworks_df["resource_type"] = homeworks_df["resource_type"].astype("category")

homework_items_df = pd.read_csv(
    SRC_DIR / "homework_items.csv",
    usecols=["id", "homework_id", "resource_type", "resource_id", "position"],
    **csv_kwargs,
)
homework_items_df["id"] = pd.to_numeric(homework_items_df["id"], errors="coerce").astype("Int32")
homework_items_df["homework_id"] = pd.to_numeric(homework_items_df["homework_id"], errors="coerce").astype("Int32")
homework_items_df["resource_id"] = pd.to_numeric(homework_items_df["resource_id"], errors="coerce").astype("Int32")
homework_items_df["position"] = pd.to_numeric(homework_items_df["position"], errors="coerce").astype("Int16")
homework_items_df["resource_type"] = homework_items_df["resource_type"].astype("category")

user_access_histories_df = pd.read_csv(
    SRC_DIR / "user_access_histories.csv",
    usecols=["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    **csv_kwargs,
)
user_access_histories_df["access_started_at"] = pd.to_datetime(user_access_histories_df["access_started_at"], format="%d %b, %Y", errors="coerce")
user_access_histories_df["access_expired_at"] = pd.to_datetime(user_access_histories_df["access_expired_at"], format="%d %b, %Y", errors="coerce")
user_access_histories_df["users_course_id"] = pd.to_numeric(user_access_histories_df["users_course_id"], errors="coerce").astype("Int32")
user_access_histories_df["activator_class"] = user_access_histories_df["activator_class"].astype("category")

user_lessons_df = pd.read_csv(
    SRC_DIR / "user_lessons.csv",
    usecols=[
        "user_id",
        "lesson_id",
        "group_id",
        "video_visited",
        "translation_visited",
        "users_course_id",
        "solved",
        "solved_tasks_count",
        "wk_points",
        "wk_solved_task_count",
    ],
    **csv_kwargs,
)
user_lessons_df["user_id"] = pd.to_numeric(user_lessons_df["user_id"], errors="coerce").astype("Int32")
user_lessons_df["lesson_id"] = pd.to_numeric(user_lessons_df["lesson_id"], errors="coerce").astype("Int32")
user_lessons_df["group_id"] = pd.to_numeric(user_lessons_df["group_id"], errors="coerce").astype("Int32")
user_lessons_df["users_course_id"] = pd.to_numeric(user_lessons_df["users_course_id"], errors="coerce").astype("Int32")
user_lessons_df["solved_tasks_count"] = pd.to_numeric(user_lessons_df["solved_tasks_count"], errors="coerce").astype("Int16")
user_lessons_df["wk_points"] = pd.to_numeric(user_lessons_df["wk_points"], errors="coerce").astype("Float32")
user_lessons_df["wk_solved_task_count"] = pd.to_numeric(user_lessons_df["wk_solved_task_count"], errors="coerce").astype("Int16")
user_lessons_df["video_visited"] = user_lessons_df["video_visited"].astype("boolean")
user_lessons_df["translation_visited"] = user_lessons_df["translation_visited"].astype("boolean")
user_lessons_df["solved"] = user_lessons_df["solved"].astype("boolean")

user_activity_histories_df = pd.read_csv(
    SRC_DIR / "user_activity_histories.csv",
    usecols=["user_lesson_id", "action", "created_at"],
    **csv_kwargs,
)
user_activity_histories_df["created_at"] = pd.to_datetime(user_activity_histories_df["created_at"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
user_activity_histories_df["user_lesson_id"] = pd.to_numeric(user_activity_histories_df["user_lesson_id"], errors="coerce").astype("Int32")
user_activity_histories_df["action"] = user_activity_histories_df["action"].astype("category")


In [4]:
# Ответы, тренинги, медиа, награды и course-level event-log.

user_answers_df = pd.read_csv(
    SRC_DIR / "user_answers.csv",
    usecols=[
        "user_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "results",
        "skipped",
        "resource_type",
        "resource_id",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    **csv_kwargs,
)
user_answers_df["submitted_at"] = pd.to_datetime(user_answers_df["submitted_at"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
user_answers_df["user_id"] = pd.to_numeric(user_answers_df["user_id"], errors="coerce").astype("Int32")
user_answers_df["task_id"] = pd.to_numeric(user_answers_df["task_id"], errors="coerce").astype("Int32")
user_answers_df["attempts"] = pd.to_numeric(user_answers_df["attempts"], errors="coerce").astype("Int8")
user_answers_df["points"] = pd.to_numeric(user_answers_df["points"], errors="coerce").astype("Float32")
user_answers_df["max_attempts"] = pd.to_numeric(user_answers_df["max_attempts"], errors="coerce").astype("Int8")
user_answers_df["resource_id"] = pd.to_numeric(user_answers_df["resource_id"], errors="coerce").astype("Int32")
user_answers_df["async_check_status"] = pd.to_numeric(user_answers_df["async_check_status"], errors="coerce").astype("Int8")
user_answers_df["solved"] = user_answers_df["solved"].astype("boolean")
user_answers_df["skipped"] = user_answers_df["skipped"].astype("boolean")
user_answers_df["wk_partial_answer"] = user_answers_df["wk_partial_answer"].astype("boolean")
user_answers_df["resource_type"] = user_answers_df["resource_type"].astype("category")

user_trainings_df = pd.read_csv(
    SRC_DIR / "user_trainings.csv",
    usecols=[
        "user_id",
        "training_id",
        "solved_tasks_count",
        "earned_points",
        "type",
        "state",
        "submitted_answers_count",
        "started_at",
        "finished_at",
        "attempts",
        "mark",
        "mark_saved_at",
    ],
    **csv_kwargs,
)
user_trainings_df["started_at"] = pd.to_datetime(user_trainings_df["started_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_trainings_df["finished_at"] = pd.to_datetime(user_trainings_df["finished_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_trainings_df["mark_saved_at"] = pd.to_datetime(user_trainings_df["mark_saved_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_trainings_df["user_id"] = pd.to_numeric(user_trainings_df["user_id"], errors="coerce").astype("Int32")
user_trainings_df["training_id"] = pd.to_numeric(user_trainings_df["training_id"], errors="coerce").astype("Int32")
user_trainings_df["solved_tasks_count"] = pd.to_numeric(user_trainings_df["solved_tasks_count"], errors="coerce").astype("Int16")
user_trainings_df["earned_points"] = pd.to_numeric(user_trainings_df["earned_points"], errors="coerce").astype("Float32")
user_trainings_df["submitted_answers_count"] = pd.to_numeric(user_trainings_df["submitted_answers_count"], errors="coerce").astype("Int16")
user_trainings_df["attempts"] = pd.to_numeric(user_trainings_df["attempts"], errors="coerce").astype("Int8")
user_trainings_df["mark"] = pd.to_numeric(user_trainings_df["mark"], errors="coerce").astype("Float32")
user_trainings_df["type"] = user_trainings_df["type"].astype("category")
user_trainings_df["state"] = user_trainings_df["state"].astype("category")

wk_users_courses_actions_df = pd.read_csv(
    SRC_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    **csv_kwargs,
)
wk_users_courses_actions_df["created_at"] = pd.to_datetime(wk_users_courses_actions_df["created_at"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
wk_users_courses_actions_df["updated_at"] = pd.to_datetime(wk_users_courses_actions_df["updated_at"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
wk_users_courses_actions_df["user_id"] = pd.to_numeric(wk_users_courses_actions_df["user_id"], errors="coerce").astype("Int32")
wk_users_courses_actions_df["users_course_id"] = pd.to_numeric(wk_users_courses_actions_df["users_course_id"], errors="coerce").astype("Int32")
wk_users_courses_actions_df["lesson_id"] = pd.to_numeric(wk_users_courses_actions_df["lesson_id"], errors="coerce").astype("Int32")
wk_users_courses_actions_df["action"] = wk_users_courses_actions_df["action"].astype("category")

wk_media_view_sessions_df = pd.read_csv(
    SRC_DIR / "wk_media_view_sessions.csv",
    usecols=["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    **csv_kwargs,
)
wk_media_view_sessions_df["started_at"] = pd.to_datetime(wk_media_view_sessions_df["started_at"], format="%d %b, %Y, %H:%M", errors="coerce")
wk_media_view_sessions_df["resource_id"] = pd.to_numeric(wk_media_view_sessions_df["resource_id"], errors="coerce").astype("Int32")
wk_media_view_sessions_df["viewer_id"] = pd.to_numeric(wk_media_view_sessions_df["viewer_id"], errors="coerce").astype("Int32")
wk_media_view_sessions_df["segments_total"] = pd.to_numeric(wk_media_view_sessions_df["segments_total"], errors="coerce").astype("Int16")
wk_media_view_sessions_df["viewed_segments_count"] = pd.to_numeric(wk_media_view_sessions_df["viewed_segments_count"], errors="coerce").astype("Int16")
wk_media_view_sessions_df["resource_type"] = wk_media_view_sessions_df["resource_type"].astype("category")
wk_media_view_sessions_df["kind"] = wk_media_view_sessions_df["kind"].astype("category")

award_badges_df = pd.read_csv(
    SRC_DIR / "award_badges.csv",
    usecols=["id", "name", "title", "level", "quota", "special"],
    **csv_kwargs,
)
award_badges_df["id"] = pd.to_numeric(award_badges_df["id"], errors="coerce").astype("Int8")
award_badges_df["level"] = pd.to_numeric(award_badges_df["level"], errors="coerce").astype("Int8")
award_badges_df["quota"] = pd.to_numeric(award_badges_df["quota"], errors="coerce").astype("Int16")
award_badges_df["name"] = award_badges_df["name"].astype("category")
award_badges_df["title"] = award_badges_df["title"].astype("category")
award_badges_df["special"] = award_badges_df["special"].astype("boolean")

user_award_badges_df = pd.read_csv(
    SRC_DIR / "user_award_badges.csv",
    usecols=["award_badge_id", "user_id", "created_at"],
    **csv_kwargs,
)
user_award_badges_df["created_at"] = pd.to_datetime(user_award_badges_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
user_award_badges_df["award_badge_id"] = pd.to_numeric(user_award_badges_df["award_badge_id"], errors="coerce").astype("Int8")
user_award_badges_df["user_id"] = pd.to_numeric(user_award_badges_df["user_id"], errors="coerce").astype("Int32")


In [5]:
# Сводные таблицы по модулям и итоговая сводка по импортированным DataFrame.

stats_module_1_df = pd.read_csv(SRC_DIR / "stats__module_1.csv", **csv_kwargs)
stats_module_1_df["Дата зачисления"] = pd.to_datetime(stats_module_1_df["Дата зачисления"], format="%Y-%m-%d", errors="coerce")
stats_module_1_df["Дата сдачи ПА (МСК)"] = pd.to_datetime(stats_module_1_df["Дата сдачи ПА (МСК)"], format="%Y-%m-%d", errors="coerce")
stats_module_1_df["user_id"] = pd.to_numeric(stats_module_1_df["user_id"], errors="coerce").astype("Int32")
stats_module_1_df["teacher_id"] = pd.to_numeric(stats_module_1_df["teacher_id"], errors="coerce").astype("Int32")
stats_module_1_df["id параллели"] = pd.to_numeric(stats_module_1_df["id параллели"], errors="coerce").astype("Int32")
stats_module_1_df["course_id"] = pd.to_numeric(stats_module_1_df["course_id"], errors="coerce").astype("Int32")
stats_module_1_df["Просмотрел уроков"] = pd.to_numeric(stats_module_1_df["Просмотрел уроков"], errors="coerce").astype("Int16")
stats_module_1_df["Просмотрено контента"] = pd.to_numeric(stats_module_1_df["Просмотрено контента"], errors="coerce").astype("Float32")
stats_module_1_df["Решено ИЗ"] = pd.to_numeric(stats_module_1_df["Решено ИЗ"], errors="coerce").astype("Int16")
stats_module_1_df["Балл ПА"] = pd.to_numeric(stats_module_1_df["Балл ПА"], errors="coerce").astype("Float32")

stats_module_2_df = pd.read_csv(SRC_DIR / "stats__module_2.csv", **csv_kwargs)
stats_module_2_df["Дата сдачи ПА (МСК)"] = pd.to_datetime(stats_module_2_df["Дата сдачи ПА (МСК)"], format="%Y-%m-%d", errors="coerce")
stats_module_2_df["user_id"] = pd.to_numeric(stats_module_2_df["user_id"], errors="coerce").astype("Int32")
stats_module_2_df["teacher_id"] = pd.to_numeric(stats_module_2_df["teacher_id"], errors="coerce").astype("Int32")
stats_module_2_df["course_id"] = pd.to_numeric(stats_module_2_df["course_id"], errors="coerce").astype("Int32")
stats_module_2_df["id параллели"] = pd.to_numeric(stats_module_2_df["id параллели"], errors="coerce").astype("Int32")
stats_module_2_df["Посмотрел уроков на 80%"] = pd.to_numeric(stats_module_2_df["Посмотрел уроков на 80%"], errors="coerce").astype("Int16")
stats_module_2_df["Просмотрено контента (ед)"] = pd.to_numeric(stats_module_2_df["Просмотрено контента (ед)"], errors="coerce").astype("Int16")
stats_module_2_df["Смотрел уроков"] = pd.to_numeric(stats_module_2_df["Смотрел уроков"], errors="coerce").astype("Int16")
stats_module_2_df["Решено ИЗ"] = pd.to_numeric(stats_module_2_df["Решено ИЗ"], errors="coerce").astype("Int16")
stats_module_2_df["Балл ПА"] = pd.to_numeric(stats_module_2_df["Балл ПА"], errors="coerce").astype("Float32")

stats_module_3_df = pd.read_csv(SRC_DIR / "stats__module_3.csv", **csv_kwargs)
stats_module_3_df["Дата сдачи ПА (МСК)"] = pd.to_datetime(stats_module_3_df["Дата сдачи ПА (МСК)"], format="%Y-%m-%d", errors="coerce")
stats_module_3_df["user_id"] = pd.to_numeric(stats_module_3_df["user_id"], errors="coerce").astype("Int32")
stats_module_3_df["teacher_id"] = pd.to_numeric(stats_module_3_df["teacher_id"], errors="coerce").astype("Int32")
stats_module_3_df["course_id"] = pd.to_numeric(stats_module_3_df["course_id"], errors="coerce").astype("Int32")
stats_module_3_df["id параллели"] = pd.to_numeric(stats_module_3_df["id параллели"], errors="coerce").astype("Int32")
stats_module_3_df["Посмотрел уроков на 80%"] = pd.to_numeric(stats_module_3_df["Посмотрел уроков на 80%"], errors="coerce").astype("Int16")
stats_module_3_df["Смотрел уроков"] = pd.to_numeric(stats_module_3_df["Смотрел уроков"], errors="coerce").astype("Int16")
stats_module_3_df["Просмотрено контента (ед)"] = pd.to_numeric(stats_module_3_df["Просмотрено контента (ед)"], errors="coerce").astype("Int16")
stats_module_3_df["Решено ИЗ"] = pd.to_numeric(stats_module_3_df["Решено ИЗ"], errors="coerce").astype("Int16")
stats_module_3_df["Балл ПА"] = pd.to_numeric(stats_module_3_df["Балл ПА"], errors="coerce").astype("Float32")

stats_module_4_df = pd.read_csv(SRC_DIR / "stats__module_4.csv", **csv_kwargs)
stats_module_4_df["user_id"] = pd.to_numeric(stats_module_4_df["user_id"], errors="coerce").astype("Int32")
stats_module_4_df["teacher_id"] = pd.to_numeric(stats_module_4_df["teacher_id"], errors="coerce").astype("Int32")
stats_module_4_df["course_id"] = pd.to_numeric(stats_module_4_df["course_id"], errors="coerce").astype("Int32")
stats_module_4_df["id параллели"] = pd.to_numeric(stats_module_4_df["id параллели"], errors="coerce").astype("Int32")
stats_module_4_df["Посмотрел уроков на 80%"] = pd.to_numeric(stats_module_4_df["Посмотрел уроков на 80%"], errors="coerce").astype("Int16")
stats_module_4_df["Смотрел уроков"] = pd.to_numeric(stats_module_4_df["Смотрел уроков"], errors="coerce").astype("Int16")
stats_module_4_df["Просмотрено контента (ед)"] = pd.to_numeric(stats_module_4_df["Просмотрено контента (ед)"], errors="coerce").astype("Int16")
stats_module_4_df["Решено ИЗ"] = pd.to_numeric(stats_module_4_df["Решено ИЗ"], errors="coerce").astype("Int16")

dataframes = {
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "lessons_df": lessons_df,
    "groups_df": groups_df,
    "trainings_df": trainings_df,
    "lesson_tasks_df": lesson_tasks_df,
    "homeworks_df": homeworks_df,
    "homework_items_df": homework_items_df,
    "user_access_histories_df": user_access_histories_df,
    "user_lessons_df": user_lessons_df,
    "user_activity_histories_df": user_activity_histories_df,
    "user_answers_df": user_answers_df,
    "user_trainings_df": user_trainings_df,
    "wk_users_courses_actions_df": wk_users_courses_actions_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
    "award_badges_df": award_badges_df,
    "user_award_badges_df": user_award_badges_df,
    "stats_module_1_df": stats_module_1_df,
    "stats_module_2_df": stats_module_2_df,
    "stats_module_3_df": stats_module_3_df,
    "stats_module_4_df": stats_module_4_df,
}

import_summary_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
        "memory_mb": [round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2) for df in dataframes.values()],
    }
)
total_memory_mb = round(import_summary_df["memory_mb"].sum(), 2)


## Аудит качества данных

Смысл блока: быстро посмотреть размер таблиц, объём памяти, типы колонок, пропуски и диапазоны значений до любой дальнейшей чистки.


In [6]:
display(import_summary_df)
print(f"Total memory: {total_memory_mb:.2f} MB")

for name, df in dataframes.items():
    print(f"\n{name}: shape={df.shape}")
    df.info(show_counts=True)
    display(df.describe(include="all").T)


,dataframe,rows,cols,memory_mb
0,users_df,95395,17,6.75
1,users_courses_df,290835,13,21.08
2,lessons_df,3369,12,0.15
3,groups_df,13076,12,0.75
4,trainings_df,410,7,0.01
5,lesson_tasks_df,29544,5,0.56
6,homeworks_df,1226,4,0.02
7,homework_items_df,5901,5,0.11
8,user_access_histories_df,667124,4,14.00
9,user_lessons_df,3070664,10,108.35


Total memory: 2746.33 MB

users_df: shape=(95395, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95395 entries, 0 to 95394
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id                            95395 non-null  Int32         
 1   last_explainer_seen_→_course  8217 non-null   Int8          
 2   created_at                    95395 non-null  datetime64[ns]
 3   updated_at                    95395 non-null  datetime64[ns]
 4   type                          95395 non-null  category      
 5   sign_in_count                 95395 non-null  Int16         
 6   last_sign_in_at               95133 non-null  datetime64[ns]
 7   grade_id                      95395 non-null  Int16         
 8   subscribed                    95395 non-null  boolean       
 9   is_teacher                    95395 non-null  boolean       
 10  timezone                      95217 non-

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,95395.0,<NA>,<NA>,<NA>,713619.057257,665740.0,689652.5,713630.0,737571.5,761603.0,27654.314533
last_explainer_seen_→_course,8217.0,<NA>,<NA>,<NA>,6.382743,1.0,7.0,7.0,7.0,7.0,1.449124
created_at,95395,NaN,NaN,NaN,2025-09-02 10:04:45.591068928,2025-01-31 14:16:00,2025-06-05 19:58:30,2025-09-30 11:31:00,2025-11-12 08:38:00,2026-03-27 16:13:00,NaN
updated_at,95395,NaN,NaN,NaN,2025-11-25 07:39:32.594789888,2025-02-17 07:30:00,2025-11-18 20:30:30,2025-12-16 12:16:00,2025-12-25 01:55:00,2026-03-27 16:47:00,NaN
type,95395,2,User::Pupil,90647,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sign_in_count,95395.0,<NA>,<NA>,<NA>,18.956077,0.0,4.0,9.0,19.0,1390.0,37.564549
last_sign_in_at,95133,NaN,NaN,NaN,2025-10-23 13:33:19.951436288,2025-02-17 06:46:00,2025-08-31 12:06:00,2025-11-25 12:30:00,2025-12-14 19:45:00,2026-03-27 16:47:00,NaN
grade_id,95395.0,<NA>,<NA>,<NA>,3009.521076,3000.0,3010.0,3010.0,3010.0,3012.0,1.935844
subscribed,95395,2,True,94940,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_teacher,95395,1,False,95395,NaN,NaN,NaN,NaN,NaN,NaN,NaN



users_courses_df: shape=(290835, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290835 entries, 0 to 290834
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   id                        290835 non-null  Int32         
 1   user_id                   290835 non-null  Int32         
 2   course_id                 290835 non-null  Int32         
 3   state                     290835 non-null  category      
 4   created_at                290835 non-null  datetime64[ns]
 5   updated_at                290835 non-null  datetime64[ns]
 6   access_finished_at        290741 non-null  datetime64[ns]
 7   wk_points                 205925 non-null  Float32       
 8   wk_max_points             290710 non-null  Float32       
 9   wk_max_viewable_lessons   290710 non-null  Float32       
 10  wk_max_task_count         290710 non-null  Float32       
 11  wk_officially_started_at  9

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,290835.0,<NA>,<NA>,<NA>,597371.301858,449032.0,522024.5,596950.0,672643.5,746383.0,86286.053626
user_id,290835.0,<NA>,<NA>,<NA>,715722.751409,665740.0,692031.0,716070.0,739958.0,761578.0,27293.035063
course_id,290835.0,<NA>,<NA>,<NA>,6916813.866577,754.0,770.0,771.0,836.0,170000688.0,29739455.750645
state,290835,2,active,287548,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,290835,NaN,NaN,NaN,2025-10-02 12:20:35.549848064,2025-02-07 11:33:00,2025-07-31 00:33:00,2025-11-05 12:25:00,2025-11-28 06:49:00,2026-03-26 20:23:00,NaN
updated_at,290835,NaN,NaN,NaN,2025-10-25 12:07:41.910774272,2025-02-19 07:00:00,2025-09-15 19:34:00,2025-11-19 13:03:00,2025-12-06 18:04:30,2026-03-27 01:49:00,NaN
access_finished_at,290741,NaN,NaN,NaN,2026-03-25 09:34:34.592850688,2025-02-16 00:00:00,2026-03-10 00:00:00,2026-05-06 00:00:00,2026-05-29 00:00:00,2026-09-26 00:00:00,NaN
wk_points,205925.0,<NA>,<NA>,<NA>,52.373169,0.0,28.0,52.299999,61.330002,492.079987,45.953308
wk_max_points,290710.0,<NA>,<NA>,<NA>,78.579185,0.0,4.0,63.0,79.0,1230.0,89.545662
wk_max_viewable_lessons,290710.0,<NA>,<NA>,<NA>,12.288298,0.0,5.0,14.0,14.0,80.0,7.222628


...

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Unnamed: 0,1955.0,NaN,NaN,NaN,977.0,0.0,488.5,977.0,1465.5,1954.0,564.504207
user_id,1955.0,<NA>,<NA>,<NA>,712124.42046,700342.0,704446.5,707740.0,717713.5,748022.0,10142.866238
Кружок,1955,71,Python. Базовый уровень. Онлайн. 2 модуль (гру...,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN
teacher_id,1955.0,<NA>,<NA>,<NA>,709844.893095,667410.0,699977.0,702887.0,720071.0,757578.0,17478.529585
course_id,1955.0,<NA>,<NA>,<NA>,952.23376,951.0,951.0,951.0,954.0,954.0,1.476561
id параллели,1955.0,<NA>,<NA>,<NA>,1247.367775,962.0,1240.0,1264.0,1284.0,1299.0,72.371152
Уровень,1955,2,Начальный,1151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Посмотрел уроков на 80%,1955.0,<NA>,<NA>,<NA>,18.028133,0.0,18.0,20.0,20.0,20.0,4.24309
Просмотрено контента (ед),1955.0,<NA>,<NA>,<NA>,819.490026,0.0,817.0,875.0,896.0,1005.0,185.71087
Просмотрено 720ед видеоконт и 80% ур,1955,2,Да,1833,NaN,NaN,NaN,NaN,NaN,NaN,NaN



stats_module_3_df: shape=(1785, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1785 entries, 0 to 1784
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   Unnamed: 0                             1785 non-null   int64         
 1   user_id                                1785 non-null   Int32         
 2   Кружок                                 1785 non-null   object        
 3   teacher_id                             1785 non-null   Int32         
 4   course_id                              1785 non-null   Int32         
 5   id параллели                           1785 non-null   Int32         
 6   Посмотрел уроков на 80%                1785 non-null   Int16         
 7   Смотрел уроков                         1785 non-null   Int16         
 8   Посетил урок в онлайне                 1785 non-null   object        
 9   Просмотрено контента (ед) 

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Unnamed: 0,1785.0,NaN,NaN,NaN,892.0,0.0,446.0,892.0,1338.0,1784.0,515.429433
user_id,1785.0,<NA>,<NA>,<NA>,711766.807843,700342.0,704386.0,707487.0,716285.0,745230.0,9985.365056
Кружок,1785,66,Python. Базовый уровень. Онлайн. 3 модуль. Гру...,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN
teacher_id,1785.0,<NA>,<NA>,<NA>,710658.77591,667410.0,699912.0,702887.0,720071.0,758538.0,18947.454659
course_id,1785.0,<NA>,<NA>,<NA>,953.223529,952.0,952.0,952.0,955.0,955.0,1.474714
id параллели,1785.0,<NA>,<NA>,<NA>,1374.57423,963.0,1362.0,1382.0,1401.0,1420.0,58.737485
Посмотрел уроков на 80%,1785.0,<NA>,<NA>,<NA>,18.656022,0.0,18.0,20.0,20.0,20.0,2.859058
Смотрел уроков,1785.0,<NA>,<NA>,<NA>,19.402241,0.0,20.0,20.0,20.0,20.0,2.419232
Посетил урок в онлайне,1785,2,Да,1435,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Просмотрено контента (ед),1785.0,<NA>,<NA>,<NA>,843.763025,0.0,829.0,873.0,895.0,982.0,119.829669



stats_module_4_df: shape=(1707, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1707 entries, 0 to 1706
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Unnamed: 0                             1707 non-null   int64 
 1   user_id                                1707 non-null   Int32 
 2   Кружок                                 1689 non-null   object
 3   teacher_id                             1689 non-null   Int32 
 4   course_id                              1689 non-null   Int32 
 5   id параллели                           1689 non-null   Int32 
 6   Посмотрел уроков на 80%                1707 non-null   Int16 
 7   Смотрел уроков                         1707 non-null   Int16 
 8   Посетил урок в онлайне                 1707 non-null   object
 9   Просмотрено контента (ед)              1707 non-null   Int16 
 10  Просмотрено 720ед видеоконт и 80% ур   1707 non

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,1707.0,NaN,NaN,NaN,853.0,492.912771,0.0,426.5,853.0,1279.5,1706.0
user_id,1707.0,<NA>,<NA>,<NA>,711773.151142,10056.422593,700342.0,704357.0,707448.0,716300.5,745230.0
Кружок,1689,2,Python. Начальный уровень. Онлайн. 4 модуль,1001,NaN,NaN,NaN,NaN,NaN,NaN,NaN
teacher_id,1689.0,<NA>,<NA>,<NA>,709000.848431,16969.857409,667410.0,699912.0,702887.0,720071.0,757578.0
course_id,1689.0,<NA>,<NA>,<NA>,954.222025,1.474455,953.0,953.0,953.0,956.0,956.0
id параллели,1689.0,<NA>,<NA>,<NA>,1563.601539,162.819138,964.0,1580.0,1595.0,1616.0,1662.0
Посмотрел уроков на 80%,1707.0,<NA>,<NA>,<NA>,1.19215,1.881136,0.0,0.0,0.0,2.0,8.0
Смотрел уроков,1707.0,<NA>,<NA>,<NA>,1.39133,2.08756,0.0,0.0,0.0,3.0,8.0
Посетил урок в онлайне,1707,2,Нет,1131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Просмотрено контента (ед),1707.0,<NA>,<NA>,<NA>,57.353837,87.976724,0.0,0.0,0.0,91.0,360.0


## Проверка на дубли

Смысл блока: проверить полные дубли строк после импорта выбранных колонок.


In [7]:
duplicates_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "duplicate_rows": [int(df.duplicated().sum()) for df in dataframes.values()],
        "rows": [df.shape[0] for df in dataframes.values()],
    }
)
duplicates_df["share_pct"] = (duplicates_df["duplicate_rows"] / duplicates_df["rows"] * 100).round(2)
display(duplicates_df.sort_values("duplicate_rows", ascending=False))


,dataframe,duplicate_rows,rows,share_pct
13,wk_users_courses_actions_df,5491157,12909207,42.54
11,user_answers_df,4950084,15176182,32.62
8,user_access_histories_df,355372,667124,53.27
10,user_activity_histories_df,33312,3031137,1.10
14,wk_media_view_sessions_df,3558,852358,0.42
19,stats_module_3_df,0,1785,0.00
18,stats_module_2_df,0,1955,0.00
17,stats_module_1_df,0,3261,0.00
16,user_award_badges_df,0,252843,0.00
15,award_badges_df,0,6,0.00


### Дубли в `stats__module_*`

Смысл блока: проверить, не связаны ли дубли будущего `target_df` с тем, что один и тот же курс встречается в разных модулях.

Выводы по текущим данным:

- дубли по `(user_id, course_id)` есть только в `stats__module_1`;
- `course_id` между `stats__module_1`, `stats__module_2`, `stats__module_3`, `stats__module_4` не пересекаются, то есть каждый курс относится только к одному модулю;
- в дублирующихся строках `stats__module_1` совпадают все признаки, кроме `teacher_id`.


In [8]:
stats_module_duplicates_df = pd.DataFrame(
    {
        "module": [1, 2, 3, 4],
        "rows": [
            stats_module_1_df.shape[0],
            stats_module_2_df.shape[0],
            stats_module_3_df.shape[0],
            stats_module_4_df.shape[0],
        ],
        "duplicate_rows_by_user_course": [
            int(stats_module_1_df.duplicated(subset=["user_id", "course_id"]).sum()),
            int(stats_module_2_df.duplicated(subset=["user_id", "course_id"]).sum()),
            int(stats_module_3_df.duplicated(subset=["user_id", "course_id"]).sum()),
            int(stats_module_4_df.duplicated(subset=["user_id", "course_id"]).sum()),
        ],
    }
)

course_module_df = pd.concat(
    [
        stats_module_1_df[["course_id"]].drop_duplicates().assign(module=1),
        stats_module_2_df[["course_id"]].drop_duplicates().assign(module=2),
        stats_module_3_df[["course_id"]].drop_duplicates().assign(module=3),
        stats_module_4_df[["course_id"]].drop_duplicates().assign(module=4),
    ],
    ignore_index=True,
)
shared_course_ids_across_modules = int(course_module_df.groupby("course_id")["module"].nunique().gt(1).sum())

duplicated_stats_module_1_df = stats_module_1_df.loc[
    stats_module_1_df.duplicated(subset=["user_id", "course_id"], keep=False)
].copy()
stats_module_1_compare_cols = [
    "user_id",
    "Кружок",
    "Дата зачисления",
    "id параллели",
    "Уровень",
    "course_id",
    "Просмотрел уроков",
    "Просмотрено контента",
    "Просмотрено 80% ур или видеоконт",
    "Посетил урок в онлайне",
    "Решено ИЗ",
    "Решены все обяз.ИЗ",
    "Пройден тек.контроль",
    "Балл ПА",
    "Сдал ПА",
    "Дата сдачи ПА (МСК)",
    "Статус",
]
all_same_except_teacher = bool(
    (
        duplicated_stats_module_1_df.groupby(["user_id", "course_id"])[stats_module_1_compare_cols]
        .nunique(dropna=False)
        .le(1)
    )
    .all()
    .all()
)

stats_module_duplicate_findings_df = pd.DataFrame(
    {
        "finding": [
            "shared course_id across stats__module_1..4",
            "all duplicate rows in stats__module_1 are identical except teacher_id",
        ],
        "value": [
            shared_course_ids_across_modules,
            all_same_except_teacher,
        ],
    }
)

display(stats_module_duplicates_df)
display(stats_module_duplicate_findings_df)


,module,rows,duplicate_rows_by_user_course
0,1,3261,270
1,2,1955,0
2,3,1785,0
3,4,1707,0


,finding,value
0,shared course_id across stats__module_1..4,0
1,all duplicate rows in stats__module_1 are iden...,True


### Sanity Checks

Смысл блока: проверить только самые базовые аномалии, которые действительно похожи на ошибки данных: перепутанный порядок дат, факт больше максимума и явные дубли по естественным ключам.


In [9]:
sanity_checks_df = pd.DataFrame(
    {
        "table": [
            "users_df",
            "users_df",
            "groups_df",
            "lessons_df",
            "lesson_tasks_df",
            "homework_items_df",
            "user_access_histories_df",
            "user_lessons_df",
            "user_lessons_df",
            "user_answers_df",
            "user_trainings_df",
            "user_trainings_df",
            "users_courses_df",
            "users_courses_df",
            "users_courses_df",
            "users_courses_df",
            "users_courses_df",
            "wk_users_courses_actions_df",
            "wk_media_view_sessions_df",
        ],
        "check": [
            "updated_at < created_at",
            "grade_changed_at < created_at",
            "wk_actual_finished_at < wk_actual_started_at",
            "lesson_number <= 0",
            "position <= 0",
            "position <= 0",
            "access_expired_at < access_started_at",
            "solved_tasks_count > wk_solved_task_count",
            "duplicate (user_id, users_course_id, lesson_id)",
            "attempts > max_attempts",
            "finished_at < started_at",
            "mark_saved_at < finished_at",
            "wk_points > wk_max_points",
            "wk_officially_started_at > access_finished_at",
            "wk_course_completed_at < created_at",
            "wk_course_completed_at < wk_officially_started_at",
            "duplicate (user_id, course_id)",
            "updated_at < created_at",
            "viewed_segments_count > segments_total",
        ],
        "bad_rows": [
            int((users_df["updated_at"] < users_df["created_at"]).sum()),
            int((users_df["grade_changed_at"] < users_df["created_at"]).sum()),
            int((groups_df["wk_actual_finished_at"] < groups_df["wk_actual_started_at"]).sum()),
            int((lessons_df["lesson_number"] <= 0).sum()),
            int((lesson_tasks_df["position"] <= 0).sum()),
            int((homework_items_df["position"] <= 0).sum()),
            int((user_access_histories_df["access_expired_at"] < user_access_histories_df["access_started_at"]).sum()),
            int((user_lessons_df["solved_tasks_count"] > user_lessons_df["wk_solved_task_count"]).sum()),
            int(user_lessons_df.duplicated(subset=["user_id", "users_course_id", "lesson_id"]).sum()),
            int((user_answers_df["attempts"] > user_answers_df["max_attempts"]).sum()),
            int((user_trainings_df["finished_at"] < user_trainings_df["started_at"]).sum()),
            int((user_trainings_df["mark_saved_at"] < user_trainings_df["finished_at"]).sum()),
            int((users_courses_df["wk_points"] > users_courses_df["wk_max_points"]).sum()),
            int((users_courses_df["wk_officially_started_at"] > users_courses_df["access_finished_at"]).sum()),
            int((users_courses_df["wk_course_completed_at"] < users_courses_df["created_at"]).sum()),
            int((users_courses_df["wk_course_completed_at"] < users_courses_df["wk_officially_started_at"]).sum()),
            int(users_courses_df.duplicated(subset=["user_id", "course_id"]).sum()),
            int((wk_users_courses_actions_df["updated_at"] < wk_users_courses_actions_df["created_at"]).sum()),
            int((wk_media_view_sessions_df["viewed_segments_count"] > wk_media_view_sessions_df["segments_total"]).sum()),
        ],
    }
)
display(sanity_checks_df)


,table,check,bad_rows
0,users_df,updated_at < created_at,0
1,users_df,grade_changed_at < created_at,0
2,groups_df,wk_actual_finished_at < wk_actual_started_at,0
3,lessons_df,lesson_number <= 0,0
4,lesson_tasks_df,position <= 0,0
5,homework_items_df,position <= 0,0
6,user_access_histories_df,access_expired_at < access_started_at,2757
7,user_lessons_df,solved_tasks_count > wk_solved_task_count,151
8,user_lessons_df,"duplicate (user_id, users_course_id, lesson_id)",0
9,user_answers_df,attempts > max_attempts,190


In [14]:
# сколько памяти занимают все датафреймы?
sum(df.memory_usage(deep=True).sum() for df in dataframes.values()) / 1024**3

np.float64(2.682870823889971)

## Найденные проблемы со связностью

- `user_activity_histories.user_lesson_id` некуда напрямую замкнуть, потому что в `user_lessons.csv` нет явного `id` или `user_lesson_id`.
- `lesson_tasks`, `homeworks` и `homework_items` связаны с остальными таблицами не полностью: часть `lesson_id`, `task_id` и `resource_id` не находится в текущих CSV.
- `groups.group_template_id` и `groups.teacher_id` живут в других пространствах ID, чем `user_lessons.group_id` и `users.id`.


## Target

Пока берём source of truth для таргета из `stats__module_1` и `stats__module_2`, потому что только в них уже есть явный столбец `Статус`.

- одна строка в `target_df` — один студент на одном модуле / курсе;
- `target = 1`, если `Статус == "Завершил"`;
- `target = 0`, если `Статус == "Отчислен"`;
- `stats__module_3` и `stats__module_4` пока не входят в `target_df`: это будущие наборы для предсказания.

Сначала собираем компактный `target_df` с уже готовым таргетом, а потом отдельно проверяем, можно ли воспроизвести этот статус по бизнес-критериям.


In [11]:
module_1_target_df = stats_module_1_df[
    [
        "user_id",
        "course_id",
        "teacher_id",
        "id параллели",
        "Уровень",
        "Кружок",
        "Дата зачисления",
        "Дата сдачи ПА (МСК)",
        "Статус",
    ]
].copy()
module_1_target_df["module"] = 1
module_1_target_df["enrolled_at"] = module_1_target_df["Дата зачисления"]
module_1_target_df["pa_submitted_at"] = module_1_target_df["Дата сдачи ПА (МСК)"]
module_1_target_df = module_1_target_df.drop(columns=["Дата зачисления", "Дата сдачи ПА (МСК)"])

module_2_target_df = stats_module_2_df[
    [
        "user_id",
        "course_id",
        "teacher_id",
        "id параллели",
        "Уровень",
        "Кружок",
        "Дата сдачи ПА (МСК)",
        "Статус",
    ]
].copy()
module_2_target_df["module"] = 2
module_2_target_df["enrolled_at"] = pd.NaT
module_2_target_df["pa_submitted_at"] = module_2_target_df["Дата сдачи ПА (МСК)"]
module_2_target_df = module_2_target_df.drop(columns=["Дата сдачи ПА (МСК)"])

target_df = pd.concat([module_1_target_df, module_2_target_df], ignore_index=True)
target_df = target_df.rename(
    columns={
        "id параллели": "group_template_id",
        "Уровень": "level",
        "Кружок": "course_name",
        "Статус": "target_status",
    }
)
target_df["target"] = target_df["target_status"].map({"Завершил": 1, "Отчислен": 0}).astype("Int8")
target_df = target_df[target_df["target"].notna()].copy()
target_df["module"] = target_df["module"].astype("Int8")
target_df["group_template_id"] = pd.to_numeric(target_df["group_template_id"], errors="coerce").astype("Int32")
target_df["level"] = target_df["level"].astype("category")
target_df["course_name"] = target_df["course_name"].astype("category")
target_df["target_status"] = target_df["target_status"].astype("category")

target_summary_df = (
    target_df.groupby(["module", "target_status"], dropna=False, observed=False)
    .size()
    .rename("rows")
    .reset_index()
)

display(target_df.head())
display(target_summary_df)
print("duplicate (user_id, course_id):", int(target_df.duplicated(subset=["user_id", "course_id"]).sum()))


,user_id,course_id,teacher_id,group_template_id,level,course_name,target_status,module,enrolled_at,pa_submitted_at,target
0,701741,1029,699869,1149,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,Завершил,1,2025-09-19,2025-12-05,1
1,737977,1029,730341,1216,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,Завершил,1,2025-11-05,2025-11-29,1
2,722851,1030,730208,1186,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,Отчислен,1,2025-10-21,NaT,0
3,709724,1030,700089,1108,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,Завершил,1,2025-09-23,2025-11-28,1
4,717763,1030,718519,1055,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,Отчислен,1,2025-10-09,NaT,0


,module,target_status,rows
0,1,Завершил,2084
1,1,Отчислен,1177
2,2,Завершил,1785
3,2,Отчислен,170


duplicate (user_id, course_id): 270


По `stats__module_1` и `stats__module_2` фактический `Статус` почти не совпадает с жёсткой формулировкой бизнес-логики, если считать обязательным посещение хотя бы одного вебинара онлайн (`groups` / `wk_media_view_sessions` -> `Посетил урок в онлайне == "Да"`). Если убрать обязательность онлайн-посещения, статус восстанавливается почти полностью: для `module_2` — без расхождений, для `module_1` — с редкими исключениями. Практически это значит, что в данных фактический таргет ближе к логике `просмотрен нужный объём контента + решены обязательные задания + пройден текущий контроль + сдана ПА (+ рефлексия для module_2)`, а критерий онлайн-посещения, вероятно, не является жёстко обязательным.

По оставшимся расхождениям в `module_1`: 6 человек имеют `Статус = Завершил`, хотя у них `Просмотрено 80% ур или видеоконт = Нет`; 7 человек имеют `Статус = Отчислен`, хотя остальные основные критерии выполнены. Это похоже либо на ручные / исключительные кейсы, либо на неконсистентное заполнение одного из флагов в `stats__module_1`.


## Повторение логики таргета

Сначала приводим `stats__module_1` и `stats__module_2` к единой схеме и собираем общий `target_logic_df`. В нём лежат исходные бизнес-поля и фактический `target_status`.


In [16]:
module_1_logic_df = stats_module_1_df[
    [
        "user_id",
        "course_id",
        "teacher_id",
        "id параллели",
        "Уровень",
        "Кружок",
        "Дата зачисления",
        "Просмотрел уроков",
        "Просмотрено контента",
        "Просмотрено 80% ур или видеоконт",
        "Посетил урок в онлайне",
        "Решено ИЗ",
        "Решены все обяз.ИЗ",
        "Пройден тек.контроль",
        "Балл ПА",
        "Сдал ПА",
        "Дата сдачи ПА (МСК)",
        "Статус",
    ]
].copy()
module_1_logic_df = module_1_logic_df.rename(
    columns={
        "id параллели": "group_template_id",
        "Уровень": "level",
        "Кружок": "course_name",
        "Дата зачисления": "enrolled_at",
        "Просмотрел уроков": "lessons_viewed",
        "Просмотрено контента": "content_viewed",
        "Просмотрено 80% ур или видеоконт": "content_threshold_status",
        "Посетил урок в онлайне": "online_attended_status",
        "Решено ИЗ": "solved_homeworks_count",
        "Решены все обяз.ИЗ": "required_homeworks_status",
        "Пройден тек.контроль": "current_control_status",
        "Балл ПА": "pa_score",
        "Сдал ПА": "pa_passed_status",
        "Дата сдачи ПА (МСК)": "pa_submitted_at",
        "Статус": "target_status",
    }
)
module_1_logic_df["module"] = 1
module_1_logic_df["reflection_status"] = pd.NA

module_2_logic_df = stats_module_2_df[
    [
        "user_id",
        "course_id",
        "teacher_id",
        "id параллели",
        "Уровень",
        "Кружок",
        "Смотрел уроков",
        "Просмотрено контента (ед)",
        "Просмотрено 720ед видеоконт и 80% ур ",
        "Посетил урок в онлайне",
        "Решено ИЗ",
        "Решены все обяз.ИЗ",
        "Пройден тек.контроль",
        "Балл ПА",
        "Сдал ПА",
        "Дата сдачи ПА (МСК)",
        "Пройдена рефлексия",
        "Статус",
    ]
].copy()
module_2_logic_df = module_2_logic_df.rename(
    columns={
        "id параллели": "group_template_id",
        "Уровень": "level",
        "Кружок": "course_name",
        "Смотрел уроков": "lessons_viewed",
        "Просмотрено контента (ед)": "content_viewed",
        "Просмотрено 720ед видеоконт и 80% ур ": "content_threshold_status",
        "Посетил урок в онлайне": "online_attended_status",
        "Решено ИЗ": "solved_homeworks_count",
        "Решены все обяз.ИЗ": "required_homeworks_status",
        "Пройден тек.контроль": "current_control_status",
        "Балл ПА": "pa_score",
        "Сдал ПА": "pa_passed_status",
        "Дата сдачи ПА (МСК)": "pa_submitted_at",
        "Пройдена рефлексия": "reflection_status",
        "Статус": "target_status",
    }
)
module_2_logic_df["module"] = 2
module_2_logic_df["enrolled_at"] = pd.NaT

target_logic_df = pd.concat([module_1_logic_df, module_2_logic_df], ignore_index=True)
target_logic_df = target_logic_df[
    [
        "module",
        "user_id",
        "course_id",
        "teacher_id",
        "group_template_id",
        "level",
        "course_name",
        "enrolled_at",
        "lessons_viewed",
        "content_viewed",
        "content_threshold_status",
        "online_attended_status",
        "solved_homeworks_count",
        "required_homeworks_status",
        "current_control_status",
        "pa_score",
        "pa_passed_status",
        "pa_submitted_at",
        "reflection_status",
        "target_status",
    ]
].copy()

display(target_logic_df.head())


,module,user_id,course_id,teacher_id,group_template_id,level,course_name,enrolled_at,lessons_viewed,content_viewed,content_threshold_status,online_attended_status,solved_homeworks_count,required_homeworks_status,current_control_status,pa_score,pa_passed_status,pa_submitted_at,reflection_status,target_status
0,1,701741,1029,699869,1149,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-09-19,20,100.0,Да,Нет,60,Да,Да,15.0,Да,2025-12-05,NaN,Завершил
1,1,737977,1029,730341,1216,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-11-05,20,97.915459,Да,Да,60,Да,Да,14.0,Да,2025-11-29,NaN,Завершил
2,1,722851,1030,730208,1186,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-21,0,0.0,Нет,Нет,0,Нет,Не сдавал,<NA>,Не сдавал,NaT,NaN,Отчислен
3,1,709724,1030,700089,1108,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-09-23,20,99.160019,Да,Да,60,Да,Да,14.0,Да,2025-11-28,NaN,Завершил
4,1,717763,1030,718519,1055,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-09,0,0.0,Нет,Нет,0,Нет,Не сдавал,<NA>,Не сдавал,NaT,NaN,Отчислен


Сохраняем `target_logic_df` в отдельный `.csv`, чтобы дальше при необходимости работать уже с одной подготовленной таблицей.


In [17]:
target_logic_df.to_csv("/Users/maria/Desktop/uni/hackathon/target_logic_df.csv", index=False)


Теперь считаем `computed_status`: это восстановленный статус по бизнес-логике без обязательного онлайн-посещения. Для `module_2` дополнительно требуется `reflection_status == "Да"`.


In [18]:
target_logic_df["computed_status"] = np.where(
    (target_logic_df["content_threshold_status"] == "Да")
    & (target_logic_df["required_homeworks_status"] == "Да")
    & (target_logic_df["current_control_status"] == "Да")
    & (target_logic_df["pa_passed_status"] == "Да")
    & ((target_logic_df["module"] == 1) | (target_logic_df["reflection_status"] == "Да")),
    "Завершил",
    "Отчислен",
)

display(target_logic_df.head())


,module,user_id,course_id,teacher_id,group_template_id,level,course_name,enrolled_at,lessons_viewed,content_viewed,...,online_attended_status,solved_homeworks_count,required_homeworks_status,current_control_status,pa_score,pa_passed_status,pa_submitted_at,reflection_status,target_status,computed_status
0,1,701741,1029,699869,1149,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-09-19,20,100.0,...,Нет,60,Да,Да,15.0,Да,2025-12-05,NaN,Завершил,Завершил
1,1,737977,1029,730341,1216,Начальный,Python. Начальный уровень. Онлайн. 1 модуль,2025-11-05,20,97.915459,...,Да,60,Да,Да,14.0,Да,2025-11-29,NaN,Завершил,Завершил
2,1,722851,1030,730208,1186,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-21,0,0.0,...,Нет,0,Нет,Не сдавал,<NA>,Не сдавал,NaT,NaN,Отчислен,Отчислен
3,1,709724,1030,700089,1108,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-09-23,20,99.160019,...,Да,60,Да,Да,14.0,Да,2025-11-28,NaN,Завершил,Завершил
4,1,717763,1030,718519,1055,Базовый,Python. Базовый уровень. Онлайн. 1 модуль,2025-10-09,0,0.0,...,Нет,0,Нет,Не сдавал,<NA>,Не сдавал,NaT,NaN,Отчислен,Отчислен


В последней ячейке считаем, насколько восстановленный `computed_status` совпадает с фактическим `target_status` по каждому модулю и в целом.


In [19]:
module_1_matches = target_logic_df[target_logic_df["module"] == 1]
module_2_matches = target_logic_df[target_logic_df["module"] == 2]

status_match_summary_df = pd.DataFrame(
    {
        "module": [1, 2, "all"],
        "rows": [
            module_1_matches.shape[0],
            module_2_matches.shape[0],
            target_logic_df.shape[0],
        ],
        "match_rows": [
            (module_1_matches["computed_status"] == module_1_matches["target_status"]).sum(),
            (module_2_matches["computed_status"] == module_2_matches["target_status"]).sum(),
            (target_logic_df["computed_status"] == target_logic_df["target_status"]).sum(),
        ],
        "match_pct": [
            round((module_1_matches["computed_status"] == module_1_matches["target_status"]).mean() * 100, 2),
            round((module_2_matches["computed_status"] == module_2_matches["target_status"]).mean() * 100, 2),
            round((target_logic_df["computed_status"] == target_logic_df["target_status"]).mean() * 100, 2),
        ],
    }
)

display(status_match_summary_df)


,module,rows,match_rows,match_pct
0,1,3261,3248,99.60
1,2,1955,1955,100.00
2,all,5216,5203,99.75
